# Gold Layer — Star Schema

Builds the Gold star schema from Silver tables.

```
silver.customers   → dim_customer
silver.products    → dim_product
silver.orders
    +              → fact_order_line
silver.order_items
order_ts range     → dim_date (2020–2030, pre-generated)
```

| Table | Grain | Key |
|---|---|---|
| `dim_customer` | One row per customer | `customer_id` |
| `dim_product` | One row per product | `product_id` |
| `dim_date` | One row per calendar day | `date_id` (YYYYMMDD) |
| `fact_order_line` | One row per order item | `order_item_id` |

## 1. Setup

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import datetime

spark.sql("USE CATALOG retail")
spark.sql("CREATE SCHEMA IF NOT EXISTS retail.gold")
print("✅ Setup complete")

## 2. Create Gold Tables (DDL)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS retail.gold.dim_customer (
    customer_id  STRING      NOT NULL,
    first_name   STRING,
    last_name    STRING,
    email        STRING,
    segment      STRING,
    city         STRING,
    country      STRING,
    is_active    BOOLEAN,
    created_at   TIMESTAMP,
    updated_at   TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail.gold.dim_product (
    product_id   STRING      NOT NULL,
    name         STRING,
    category     STRING,
    subcategory  STRING,
    unit_price   DOUBLE,
    cost_price   DOUBLE,
    is_active    BOOLEAN,
    created_at   TIMESTAMP,
    updated_at   TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail.gold.dim_date (
    date_id        INT         NOT NULL,
    full_date      DATE,
    year           INT,
    quarter        INT,
    month          INT,
    month_name     STRING,
    week_of_year   INT,
    day_of_month   INT,
    day_of_week    INT,
    day_name       STRING,
    is_weekend     BOOLEAN,
    is_month_end   BOOLEAN
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail.gold.fact_order_line (
    order_item_id  STRING      NOT NULL,
    order_id       STRING,
    date_id        INT,
    customer_id    STRING,
    product_id     STRING,
    order_status   STRING,
    shipping_city  STRING,
    quantity       LONG,
    unit_price     DOUBLE,
    line_total     DOUBLE,
    cost_price     DOUBLE,
    gross_margin   DOUBLE,
    order_ts       TIMESTAMP,
    _extracted_at  TIMESTAMP
) USING DELTA
""")

print("✅ All gold tables created (or already exist)")

## 3. Reusable Functions

In [0]:
def upsert_to_gold(df_source, gold_table, primary_key):
    """
    SCD Type 1 MERGE into a Gold table.
    - Match found  → overwrite all columns
    - No match     → insert new row
    """
    update_mapping = {col: f"source.{col}" for col in df_source.columns if col != primary_key}
    insert_mapping = {col: f"source.{col}" for col in df_source.columns}
    delta_table = DeltaTable.forName(spark, gold_table)

    (
        delta_table.alias("target")
        .merge(
            df_source.alias("source"),
            f"target.{primary_key} = source.{primary_key}"
        )
        .whenMatchedUpdate(set=update_mapping)
        .whenNotMatchedInsert(values=insert_mapping)
        .execute()
    )


def validate_gold(gold_table, primary_key, label):
    """
    Basic quality checks on a gold table:
    1. Duplicates on primary key
    2. Null count on primary key
    3. Row count
    """
    #df = spark.table(gold_table).cache()  Useful if you're running in a cluster
    # it is unnecesary if you are running in serverless compute
    df = spark.table(gold_table)

    print(f"\n{'='*50}")
    print(f" Validation — {label}")
    print(f"{'='*50}")

    # Duplicates
    dups = df.groupBy(primary_key).count().filter("count > 1").count()
    print(f"Duplicates on {primary_key}: {dups} {'⚠️  WARNING' if dups > 0 else '✅ OK'}")

    # Nulls on PK
    nulls = df.filter(F.col(primary_key).isNull()).count()
    print(f"Nulls on {primary_key}: {nulls} {'⚠️  WARNING' if nulls > 0 else '✅ OK'}")

    # Row count
    print(f"Total rows: {df.count()}")
    
    
    print("\nSample (5 rows):")
    display(df.limit(5))

    #df.unpersist() if you use .cache()


def validate_foreign_keys(fact_table, fk_rules, label="FACT"):
    """
    fk_rules: list of dicts, each:
      {
        "fact_key": "customer_id",
        "dim_table": "retail.gold.dim_customer",
        "dim_key": "customer_id",
        "rule_name": "customer_fk"
      }
    """
    print(f"\n{'='*50}")
    print(f" FK Validation — {label}")
    print(f"{'='*50}")

    for rule in fk_rules:
        fact_key = rule["fact_key"]
        dim_table = rule["dim_table"]
        dim_key = rule["dim_key"]
        rule_name = rule.get("rule_name", f"{fact_key}->{dim_table}.{dim_key}")

        q = f"""
        SELECT COUNT(*) AS orphan_count
        FROM {fact_table} f
        LEFT JOIN {dim_table} d
          ON f.{fact_key} = d.{dim_key}
        WHERE d.{dim_key} IS NULL
        """
        orphan_count = spark.sql(q).collect()[0]["orphan_count"]
        status = "OK" if orphan_count == 0 else "WARNING"
        print(f"{rule_name}: {orphan_count} orphan(s) [{status}]")

print("✅ Functions loaded")

## 4. Build `dim_date`
Pre-generated fixed range 2020-01-01 → 2030-12-31 (~3,650 rows).
`date_id` uses YYYYMMDD INT format — industry standard.

In [0]:
# Generate date range as a sequence of days
start_date = datetime.date(2020, 1, 1)
end_date   = datetime.date(2030, 12, 31)
total_days = (end_date - start_date).days + 1

df_date = (
    spark.range(total_days)
    .withColumn("full_date", F.date_add(F.lit(str(start_date)), F.col("id").cast("int")))
    .withColumn("date_id",       F.date_format(F.col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("year",          F.year("full_date"))
    .withColumn("quarter",       F.quarter("full_date"))
    .withColumn("month",         F.month("full_date"))
    .withColumn("month_name",    F.date_format(F.col("full_date"), "MMMM"))
    .withColumn("week_of_year",  F.weekofyear("full_date"))
    .withColumn("day_of_month",  F.dayofmonth("full_date"))
    .withColumn("day_of_week",   F.dayofweek("full_date"))   # 1=Sunday, 7=Saturday
    .withColumn("day_name",      F.date_format(F.col("full_date"), "EEEE"))
    .withColumn("is_weekend",    F.dayofweek(F.col("full_date")).isin([1, 7]))
    .withColumn("is_month_end",  F.col("full_date") == F.last_day(F.col("full_date")))
    .drop("id")
)

# Overwrite — dim_date is static and fully rebuilt each time
(
    df_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail.gold.dim_date")
)

print(f"✅ dim_date built — {df_date.count()} rows (2020–2030)")
validate_gold("retail.gold.dim_date", "date_id", "DIM_DATE")

## 5. Build `dim_customer`
SCD Type 1 — overwrite on match. `is_active = NOT is_deleted`.

In [0]:
df_dim_customer = (
    spark.table("retail.silver.customers")
    .select(
        F.col("customer_id"),
        F.col("first_name"),
        F.col("last_name"),
        F.col("email"),
        F.col("segment"),
        F.col("city"),
        F.col("country"),
        (~F.col("is_deleted")).alias("is_active"),
        F.col("created_at"),
        F.col("updated_at")
    )
)

upsert_to_gold(df_dim_customer, "retail.gold.dim_customer", "customer_id")
print("✅ dim_customer upserted")
validate_gold("retail.gold.dim_customer", "customer_id", "DIM_CUSTOMER")

## 6. Build `dim_product`
SCD Type 1. Keeps `category` and `subcategory` (star schema — no separate dim_category).
`unit_price` here is the **current price** — price at time of sale lives in the fact.

In [0]:
df_dim_product = (
    spark.table("retail.silver.products")
    .select(
        F.col("product_id"),
        F.col("name"),
        F.col("category"),
        F.col("subcategory"),
        F.col("unit_price"),
        F.col("cost_price"),
        (~F.col("is_deleted")).alias("is_active"),
        F.col("created_at"),
        F.col("updated_at")
    )
)

upsert_to_gold(df_dim_product, "retail.gold.dim_product", "product_id")
print("✅ dim_product upserted")
validate_gold("retail.gold.dim_product", "product_id", "DIM_PRODUCT")

## 7. Build `fact_order_line`

**Grain:** one row per `order_item_id`

**JOINs:**
- `order_items` → `orders` on `order_id` (to get `order_status`, `shipping_city`, `order_ts`, `customer_id`)
- `order_items` → `products` on `product_id` (to get `cost_price` for `gross_margin`)

**Filters:**
- `order_items.is_deleted = false`
- `orders.is_deleted = false`

**Derived columns:**
- `date_id = YYYYMMDD from order_ts`
- `gross_margin = line_total - (quantity * cost_price)`

In [0]:
df_order_items = (
    spark.table("retail.silver.order_items")
    .filter(~F.col("is_deleted"))
    .select(
        "order_item_id",
        "order_id",
        "product_id",
        "quantity",
        F.col("unit_price").alias("sale_unit_price"),  # price at time of sale
        "line_total",
        "_extracted_at"
    )
)

df_orders = (
    spark.table("retail.silver.orders")
    .filter(~F.col("is_deleted"))
    .select(
        "order_id",
        "customer_id",
        F.col("order_status"),
        "shipping_city",
        "order_ts"
    )
)

df_products = (
    spark.table("retail.silver.products")
    .select(
        "product_id",
        F.col("cost_price").alias("product_cost_price")
    )
)

# JOIN order_items + orders + products
df_fact = (
    df_order_items
    .join(df_orders,   on="order_id",   how="inner")
    .join(df_products, on="product_id", how="left")   # left: keep items even if product was deleted
    .withColumn(
        "date_id",
        F.date_format(F.col("order_ts"), "yyyyMMdd").cast("int")
    )
    .withColumn(
        "gross_margin",
        F.round(
            F.col("line_total") - (F.col("quantity") * F.col("product_cost_price")),
            2
        )
    )
    .select(
        F.col("order_item_id"),
        F.col("order_id"),
        F.col("date_id"),
        F.col("customer_id"),
        F.col("product_id"),
        F.col("order_status"),
        F.col("shipping_city"),
        F.col("quantity"),
        F.col("sale_unit_price").alias("unit_price"),
        F.col("line_total"),
        F.col("product_cost_price").alias("cost_price"),
        F.col("gross_margin"),
        F.col("order_ts"),
        F.col("_extracted_at")
    )
)

upsert_to_gold(df_fact, "retail.gold.fact_order_line", "order_item_id")
print("✅ fact_order_line upserted")
validate_gold("retail.gold.fact_order_line", "order_item_id", "FACT_ORDER_LINE")

In [0]:
fk_rules = [
    {
        "fact_key": "customer_id",
        "dim_table": "retail.gold.dim_customer",
        "dim_key": "customer_id",
        "rule_name": "customer_fk"
    },
    {
        "fact_key": "product_id",
        "dim_table": "retail.gold.dim_product",
        "dim_key": "product_id",
        "rule_name": "product_fk"
    },
    {
        "fact_key": "date_id",
        "dim_table": "retail.gold.dim_date",
        "dim_key": "date_id",
        "rule_name": "date_fk"
    }
]

validate_foreign_keys("retail.gold.fact_order_line", fk_rules, "FACT_ORDER_LINE")

In [0]:
# Compact and cluster fact table — most impactful for BI query performance
spark.sql("OPTIMIZE retail.gold.fact_order_line ZORDER BY (date_id, customer_id)")
print("✅ fact_order_line optimized — Z-ordered by (date_id, customer_id)")

# Dimensions are small — OPTIMIZE only, no ZORDER needed
spark.sql("OPTIMIZE retail.gold.dim_customer")
spark.sql("OPTIMIZE retail.gold.dim_product")
print("✅ dim_customer and dim_product optimized")

In [0]:
df_fact_gold = spark.table("retail.gold.fact_order_line")

bad_metrics = (
    df_fact_gold
    .filter(
        (F.col("quantity") < 0) |
        (F.col("unit_price") < 0) |
        (F.col("line_total") < 0)
    )
    .count()
)

print(f"Negative metric rows: {bad_metrics} {'WARNING' if bad_metrics > 0 else 'OK'}")

In [0]:
# 1) Revenue and margin by segment and month
spark.sql("""
CREATE OR REPLACE VIEW retail.gold.vw_revenue_margin_by_segment_month AS
SELECT
    d.year,
    d.month,
    c.segment,
    SUM(f.line_total) AS revenue,
    SUM(f.gross_margin) AS gross_margin,
    COUNT(DISTINCT f.order_id) AS orders
FROM retail.gold.fact_order_line f
JOIN retail.gold.dim_customer c ON f.customer_id = c.customer_id
JOIN retail.gold.dim_date d ON f.date_id = d.date_id
GROUP BY d.year, d.month, c.segment
""")

# 2) Top products by revenue
spark.sql("""
CREATE OR REPLACE VIEW retail.gold.vw_top_products_by_revenue AS
SELECT
    p.product_id,
    p.name,
    p.category,
    p.subcategory,
    SUM(f.quantity) AS units_sold,
    SUM(f.line_total) AS revenue,
    SUM(f.gross_margin) AS gross_margin
FROM retail.gold.fact_order_line f
JOIN retail.gold.dim_product p ON f.product_id = p.product_id
GROUP BY p.product_id, p.name, p.category, p.subcategory

""")

# 3) Order status funnel
spark.sql("""
CREATE OR REPLACE VIEW retail.gold.vw_order_status_funnel AS
SELECT
    order_status,
    COUNT(DISTINCT order_id) AS orders,
    SUM(line_total) AS revenue
FROM retail.gold.fact_order_line
GROUP BY order_status
""")

# 4) Customer lifetime value (simple)
spark.sql("""
CREATE OR REPLACE VIEW retail.gold.vw_customer_ltv AS
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.segment,
    COUNT(DISTINCT f.order_id) AS total_orders,
    SUM(f.line_total) AS lifetime_revenue,
    SUM(f.gross_margin) AS lifetime_margin
FROM retail.gold.fact_order_line f
JOIN retail.gold.dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.segment
""")


# 5) Daily KPI trend
spark.sql("""
CREATE OR REPLACE VIEW retail.gold.vw_daily_kpis AS
SELECT
    d.full_date,
    COUNT(DISTINCT f.order_id) AS orders,
    COUNT(DISTINCT f.customer_id) AS active_customers,
    SUM(f.line_total) AS revenue,
    SUM(f.gross_margin) AS gross_margin,
    SUM(f.quantity) AS units
FROM retail.gold.fact_order_line f
JOIN retail.gold.dim_date d ON f.date_id = d.date_id
GROUP BY d.full_date
""")

In [0]:
print("\n" + "="*60)
print(" VIEW SAMPLES")
print("="*60)

# 1) Revenue and margin by segment and month
print("\n1️⃣ Revenue & Margin by Segment & Month (Top 10)")
display(spark.table("retail.gold.vw_revenue_margin_by_segment_month").orderBy(F.desc("revenue")).limit(10))

# 2) Top products by revenue
print("\n2️⃣ Top Products by Revenue (Top 10)")
display(spark.table("retail.gold.vw_top_products_by_revenue").orderBy(F.desc("revenue")).limit(10))

# 3) Order status funnel
print("\n3️⃣ Order Status Funnel")
display(spark.table("retail.gold.vw_order_status_funnel").orderBy(F.desc("revenue")))

# 4) Customer lifetime value (Top 10)
print("\n4️⃣ Customer Lifetime Value (Top 10 Customers)")
display(spark.table("retail.gold.vw_customer_ltv").orderBy(F.desc("lifetime_revenue")).limit(10))

# 5) Daily KPIs (Last 30 days)
print("\n5️⃣ Daily KPIs (Last 30 Days)")
display(spark.table("retail.gold.vw_daily_kpis").orderBy(F.desc("full_date")).limit(30))

print("\n" + "="*60)
print("✅ All KPI views validated")
print("="*60)

## 8. Pipeline Summary

In [0]:
tables = [
    ("retail.gold.dim_date",        "date_id"),
    ("retail.gold.dim_customer",    "customer_id"),
    ("retail.gold.dim_product",     "product_id"),
    ("retail.gold.fact_order_line", "order_item_id"),
]

print("\n" + "="*50)
print(" Gold Layer — Final Summary")
print("="*50)
for table, pk in tables:
    count = spark.table(table).count()
    print(f"{table.split('.')[-1]:<25} → {count:>8} rows")
print("="*50)
print("✅ Gold layer pipeline complete")